In [ ]:
import cv2
import os

video_path = r"D:\Data\Game Dataset\az_recorder_20260103_125453.mp4"
output_dir = r"D:\Data\Game Dataset\az_recorder_20260103_125453s"

os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    filename = os.path.join(output_dir, f"frame_{frame_idx:06d}.jpg")
    cv2.imwrite(filename, frame)
    frame_idx += 1

cap.release()

print(f"Extracted {frame_idx} frames")


Extracted 8439 frames


In [2]:
import cv2
import os

video_path = r"D:\Data\Game Dataset\az_recorder_20260103_125453.mp4"
output_dir = r"D:\Data\Game Dataset\az_recorder_20260103_125453s"
os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    timestamp_us = int((frame_idx / fps) * 1_000_000)

    filename = os.path.join(
        output_dir,
        f"frame_{frame_idx:06d}_t_{timestamp_us:012d}us.jpg"
    )

    cv2.imwrite(filename, frame)
    frame_idx += 1

cap.release()

print(f"Extracted {frame_idx} frames")


Extracted 151485 frames


In [1]:
import shutil
from pathlib import Path
from PIL import Image


def pixel_meets_min(rgb, min_rgb):
    return (rgb[0] >= min_rgb[0]) and (rgb[1] >= min_rgb[1]) and (rgb[2] >= min_rgb[2])


def pixel_meets_max(rgb, max_rgb):
    return (rgb[0] <= max_rgb[0]) and (rgb[1] <= max_rgb[1]) and (rgb[2] <= max_rgb[2])


def copy_images_by_pixel_rules(
    src_dir: str,
    dst_dir: str,
    white_points,
    white_value=(220, 220, 220),
    black_points=None,
    black_value=(70, 70, 70),
):
    """
    Copy images from src_dir to dst_dir if:
      - All pixels at white_points are >= white_value
      - All pixels at black_points are <= black_value

    Points are (x, y) in image pixel coordinates.
    """

    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    img_paths = sorted([p for p in src_dir.iterdir() if p.suffix.lower() in valid_exts], key=lambda p: p.name)
    if not img_paths:
        print("No images found.")
        return

    black_points = black_points or []

    copied = 0
    checked = 0

    for img_path in img_paths:
        checked += 1
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                w, h = img.size
                px = img.load()

                # Check bounds and white points
                white_ok = True
                for (x, y) in white_points:
                    if x < 0 or x >= w or y < 0 or y >= h:
                        white_ok = False
                        break
                    rgb = px[x, y]
                    if not pixel_meets_min(rgb, white_value):
                        white_ok = False
                        break

                if not white_ok:
                    continue

                # Check bounds and black points
                black_ok = True
                for (x, y) in black_points:
                    if x < 0 or x >= w or y < 0 or y >= h:
                        black_ok = False
                        break
                    rgb = px[x, y]
                    if not pixel_meets_max(rgb, black_value):
                        black_ok = False
                        break

                if not black_ok:
                    continue

                shutil.copy2(img_path, dst_dir / img_path.name)
                copied += 1

        except Exception as e:
            print(f"Failed on {img_path.name}: {e}")

    print(f"Checked {checked} images, copied {copied} into {dst_dir}")


In [8]:
white_points = [(704, 23), (716, 22)]
white_value = (210, 210, 210)

black_points = None # [(702, 8), (1063, 37)]
black_value = (70, 70, 70)

copy_images_by_pixel_rules(
    src_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s",
    dst_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_New",
    white_points=white_points,
    white_value=white_value,
    black_points=black_points,
    black_value=black_value,
)


Checked 151485 images, copied 115996 into D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_New


In [4]:
from pathlib import Path
from PIL import Image


def resize_images_in_folder(
    src_dir: str,
    dst_dir: str,
    size: tuple[int, int],
    keep_aspect: bool = False,
):
    """
    Resize all images in src_dir to `size` (width, height)
    and save them to dst_dir.

    Args:
        src_dir: source folder with images
        dst_dir: destination folder
        size: (width, height)
        keep_aspect: if True, preserves aspect ratio with padding
    """

    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = sorted([p for p in src_dir.iterdir() if p.suffix.lower() in valid_exts])

    if not image_paths:
        print("No images found.")
        return

    for img_path in image_paths:
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")

                if keep_aspect:
                    img = img.copy()
                    img.thumbnail(size, Image.BILINEAR)

                    # Pad to exact size
                    new_img = Image.new("RGB", size, (0, 0, 0))
                    x = (size[0] - img.width) // 2
                    y = (size[1] - img.height) // 2
                    new_img.paste(img, (x, y))
                    img = new_img
                else:
                    img = img.resize(size, Image.BILINEAR)

                img.save(dst_dir / img_path.name)

        except Exception as e:
            print(f"Failed on {img_path.name}: {e}")

    print(f"Resized {len(image_paths)} images into {dst_dir}")


In [9]:
resize_images_in_folder(
    src_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay",
    dst_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_224x224",
    size=(224, 224),
    keep_aspect=True,  # set True if you want padding instead of distortion
)


Resized 115996 images into D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_224x224


In [10]:
import csv
from pathlib import Path


def sync_csv_with_folder(
    csv_path: str,
    images_dir: str,
    out_csv_path: str | None = None,
):
    """
    Remove rows from CSV whose filenames no longer exist in images_dir.

    CSV must have column: 'filename'
    Keeps original row order.
    """

    csv_path = Path(csv_path)
    images_dir = Path(images_dir)

    if out_csv_path is None:
        out_csv_path = csv_path  # overwrite
    else:
        out_csv_path = Path(out_csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    if not images_dir.exists():
        raise FileNotFoundError(f"Images dir not found: {images_dir}")

    # Collect existing filenames in folder
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    existing_files = {
        p.name for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in valid_exts
    }

    kept_rows = []

    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames

        if "filename" not in fieldnames:
            raise ValueError("CSV must contain a 'filename' column")

        for row in reader:
            if row["filename"] in existing_files:
                kept_rows.append(row)

    # Write updated CSV
    with open(out_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(kept_rows)

    print(
        f"CSV sync complete\n"
        f"  Original rows: {sum(1 for _ in open(csv_path)) - 1}\n"
        f"  Kept rows: {len(kept_rows)}\n"
        f"  Output CSV: {out_csv_path}"
    )


In [11]:
sync_csv_with_folder(
    csv_path=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_224x224.csv",
    images_dir=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_224x224",
    out_csv_path=r"D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_224x224_synced.csv",
)


CSV sync complete
  Original rows: 151485
  Kept rows: 115996
  Output CSV: D:\Data\Game Dataset\az_recorder_20260103_125453s_GamePlay_224x224_synced.csv
